# Análisis V2 - Regresión Polinomial y Regularización: Producción de Arroz

## 📌 Contexto

Este notebook documenta la **V2** del modelo de Regresión Polinomial + Regularización (Ridge, Lasso, ElasticNet) para predecir la **Producción (qq)** de arroz a partir de la **Superficie (Ha)**.

### Problema detectado en V1
- R² ~0.82 pero curvas polinomiales que se flexionaban irrealmente para "alcanzar" el outlier extremo.
- Señal clara de **overfitting** hacia datos anómalos.

### Hipótesis V2
1. Filtrar outliers mejorará la generalización.
2. Transformar a log linealizará la relación y reducirá el overfitting.
3. Optimizar `alpha` con GridSearchCV dará el mejor balance sesgo-varianza.

## 1. Metodología

Se evaluaron **5 modelos × 4 escenarios** = 20 combinaciones:

**Modelos:** Lineal, Polinomial (grado 4), Ridge, Lasso, ElasticNet.

**Escenarios:**
- A) Original (sin filtro, sin log)
- B) Filtrado (< 3000 Ha)
- C) Original + Log
- D) Filtrado + Log

Además, se aplicó **GridSearchCV (5-fold)** sobre los modelos regularizados para encontrar el `alpha` óptimo.

## 2. Resultados

### 2.1 Tabla comparativa completa

| Escenario | Modelo | R² | RMSE | MAE | MAPE (%) |
|---|---|---|---|---|---|
| A) Original | Lineal | 0.4155 | 6412.80 | 3006.52 | 423.91 |
| A) Original | Polinomial | 0.8193 | 3566.07 | 1134.59 | 69.18 |
| A) Original | Ridge | 0.8079 | 3676.77 | 1338.84 | 117.75 |
| A) Original | Lasso | 0.8193 | 3565.69 | 1134.01 | 68.71 |
| A) Original | Elastic Net | **0.8195** | **3563.34** | 1222.78 | 92.88 |
| B) Filtrado | Lineal | 0.4761 | 6716.74 | 1882.11 | 194.56 |
| B) Filtrado | Polinomial | -0.1581 | 9986.51 | 1744.17 | 69.62 |
| B) Filtrado | Ridge | 0.0542 | 9025.16 | 1639.53 | 75.19 |
| B) Filtrado | Lasso | -0.1525 | 9962.33 | 1740.16 | 69.25 |
| B) Filtrado | Elastic Net | 0.2876 | 7832.52 | 1475.33 | 66.99 |
| C) Original + Log | Lineal | 0.6989 | 4602.38 | 1390.01 | 56.86 |
| C) Original + Log | Polinomial | 0.7848 | 3890.79 | 1225.67 | 56.10 |
| C) Original + Log | **Ridge** | **0.8198** | **3560.34** | **1157.91** | 57.35 |
| C) Original + Log | Lasso | 0.7181 | 4453.42 | 1403.60 | 54.85 |
| C) Original + Log | Elastic Net | 0.7762 | 3968.28 | 1309.61 | 55.28 |
| D) Filtrado + Log | Lineal | 0.6787 | 5259.79 | 1470.68 | **49.03** |
| D) Filtrado + Log | Polinomial | 0.1277 | 8667.38 | 1654.94 | 52.78 |
| D) Filtrado + Log | Ridge | 0.6812 | 5239.93 | 1265.94 | 49.41 |
| D) Filtrado + Log | Lasso | 0.6665 | 5359.15 | 1446.99 | 49.14 |
| D) Filtrado + Log | Elastic Net | 0.5472 | 6244.58 | 1474.98 | 51.38 |

### 2.2 Mejor modelo por escenario (según R²)

| Escenario | Mejor modelo | R² | MAPE (%) |
|---|---|---|---|
| A) Original | Elastic Net | 0.8195 | 92.88 |
| B) Filtrado | Lineal | 0.4761 | 194.56 |
| C) Original + Log | Ridge | **0.8198** | 57.35 |
| D) Filtrado + Log | Ridge | 0.6812 | **49.41** |

### 2.3 GridSearchCV - Alphas óptimos

| Modelo | Alpha óptimo | R² (CV) |
|---|---|---|
| Ridge | 0.01 | **0.8307** |
| Lasso | 0.001 | 0.8270 |
| ElasticNet | alpha=0.01, l1_ratio=0.8 | 0.8268 |

> Ridge con alpha=0.01 alcanza el mejor R² de validación cruzada: **0.8307**.

## 3. Interpretación

### 🔴 Escenario A: overfitting clásico
El polinomial y sus variantes regularizadas llegan a R²~0.82 con datos crudos, pero el MAPE sigue entre 69% y 118%. Están **memorizando el outlier**.

### 🟡 Escenario B: filtrado solo = catástrofe
Los modelos polinomiales tienen **R² negativo** (-0.15). Esto ocurre cuando el modelo es **peor que predecir la media**. Motivo: sin log, los datos filtrados no siguen una curva polinomial, siguen una nube dispersa.

### 🔵 Escenario C: log rescata todo
Con log, Ridge alcanza **R²=0.8198** y reduce MAPE a 57%. Es el mejor R² de todos los escenarios, pero el MAPE aún no es ideal.

### 🟢 Escenario D: la mejor combinación para producción real 🏆
Con filtrado + log:
- Ridge logra **R²=0.6812** (más bajo que C) pero con **MAPE=49.41%** (el más bajo).
- **Trade-off**: sacrificamos un poco de R² por un modelo más honesto con la mayoría de los productores.

### 🏅 Ridge es el rey
En **los 2 escenarios con log**, Ridge gana. Es más estable que Lasso/ElasticNet cuando hay colinealidad y outliers residuales.

## 4. Visualización

![Regresión Polinomial V2](../03_imagenes/25_regresion_polinomial_v2.png)

**Observaciones:**
- En A) las curvas se disparan hacia arriba intentando alcanzar el outlier.
- En B) las curvas oscilan sin sentido (overfitting sin estructura).
- En C) y D) las curvas son suaves y coherentes con la relación real.

## 5. Comparación con Regresión Lineal V2

| Modelo | Mejor R² | Mejor MAPE | Escenario |
|---|---|---|---|
| Lineal V2 (08) | 0.6898 | 49.35% | D) Filtrado + Log |
| Polinomial V2 (09) | **0.8198** | 57.35% | C) Original + Log |
| Polinomial V2 (09) | 0.6812 | **49.41%** | D) Filtrado + Log |

### Conclusión del duelo
- **Si priorizamos R²**: Polinomial Ridge en C gana (0.8198 vs 0.6898).
- **Si priorizamos MAPE**: prácticamente empate (49.35% vs 49.41%).
- **Si priorizamos simplicidad**: Lineal V2-D es más interpretable y casi igual de preciso.

> 🎯 **Recomendación**: usar **Lineal V2-D** para producción (simple + interpretable), y **Polinomial Ridge V2-C** como modelo de referencia de mayor capacidad.

## 6. Conclusiones y próximos pasos

### ✅ Aprendizajes
1. **Los polinomios sin log son peligrosos**: capturan outliers y producen R² negativos al filtrar.
2. **El log es el mejor amigo del polinomio**: transforma una relación multiplicativa en aditiva.
3. **Ridge con alpha bajo (0.01)** es el regularizador más robusto en este dataset.
4. **La regularización sola no resuelve el problema de outliers**; hay que atacar los datos primero.

### 🔜 Próximos pasos
1. Incorporar `RENDIMIENTO (Kg/Ha)` como segunda variable predictora.
2. Probar **Random Forest** y **Gradient Boosting** con las mismas estrategias (V2).
3. Explorar umbrales alternativos: 2000 Ha, 1500 Ha.
4. Aplicar validación cruzada con `RepeatedKFold` para confirmar estabilidad.
5. Documentar todo en la bitácora con versiones v2.0 de cada modelo.